# CareArm YOLO 학습 노트북

물체 3종(`tissue_core`, `candy_tin`, `cube_block`)을 YOLO11n으로 학습해요.

**순서**: 위에서부터 셀을 하나씩 실행(▶)하세요.

시작 전에 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**로 바꿔 주세요.

## 1. 준비 (설치 + GPU 확인)

In [ ]:
!pip -q install ultralytics
import torch, ultralytics
print('GPU 사용 가능:', torch.cuda.is_available())
ultralytics.checks()

## 2. 데이터셋 올리기
Roboflow에서 **YOLOv11 형식**으로 내려받은 zip 파일을 선택하세요.

In [ ]:
from google.colab import files
import zipfile, os, glob, shutil
up = files.upload()
zip_name = list(up.keys())[0]
shutil.rmtree('/content/data', ignore_errors=True)
zipfile.ZipFile(zip_name).extractall('/content/data')
for sp in ['train','valid','test']:
    n = len(glob.glob(f'/content/data/{sp}/images/*'))
    print(f'{sp}: {n}장')

## 3. data.yaml 경로 정리
Roboflow가 만든 `data.yaml`의 경로를 Colab 경로로 고쳐요.

In [ ]:
import yaml
y = yaml.safe_load(open('/content/data/data.yaml'))
y['path'] = '/content/data'
y['train'] = 'train/images'
y['val'] = 'valid/images'
y['test'] = 'test/images'
yaml.safe_dump(y, open('/content/data/data.yaml','w'), allow_unicode=True)
print('클래스:', y['names'])

## 4. 학습
- 모델: YOLO11n (친구 노트북에서 쓰는 것과 같은 크기)
- 이미지 크기: 320 (모듈캠 320×240에 맞춤)
- GPU에서 보통 몇 분 걸려요.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
results = model.train(
    data='/content/data/data.yaml',
    imgsz=320,
    epochs=100,
    patience=30,      # 30에폭 동안 나아지지 않으면 일찍 멈춤
    batch=16,
    fliplr=0.5,       # 좌우 뒤집기만 가볍게
    name='carearm',
    exist_ok=True,
)

## 5. 평가 (Test 사진, 학습에 안 쓴 사진)
- **mAP50**: 1에 가까울수록 좋아요.
- 클래스별 결과와 **혼동 행렬**(서로 헷갈렸는지)을 확인하세요.

In [ ]:
best = YOLO('/content/runs/detect/carearm/weights/best.pt')
m = best.val(data='/content/data/data.yaml', split='test', imgsz=320, name='carearm_test', exist_ok=True)
print('Test mAP50   :', round(float(m.box.map50), 3))
print('Test mAP50-95:', round(float(m.box.map), 3))
for k, ci in enumerate(m.box.ap_class_index):
    print(f'  {best.names[int(ci)]:12s} mAP50 = {float(m.box.ap50[k]):.3f}')
from IPython.display import Image, display
display(Image('/content/runs/detect/carearm_test/confusion_matrix.png', width=500))

## 6. Test 사진에 직접 돌려 보기
박스, 라벨, 확신도를 눈으로 확인해요. 확신도가 대부분 몇 이상인지 보면 **"이 값 이상이면 집는다"** 기준을 정할 수 있어요.

In [ ]:
res = best.predict('/content/data/test/images', imgsz=320, conf=0.25, save=True, name='carearm_pred', exist_ok=True, verbose=False)
for r in res:
    name = os.path.basename(r.path)
    for b in r.boxes:
        cx, cy = b.xywh[0][:2].tolist()
        print(f'{name[:28]:28s} {best.names[int(b.cls)]:12s} 확신도 {float(b.conf):.2f}  중심 ({cx:.0f}, {cy:.0f})')
for p in sorted(glob.glob('/content/runs/detect/carearm_pred/*'))[:8]:
    display(Image(p, width=320))

## 7. 모델 파일 내려받기
`best.pt`를 내려받아 팀 GitHub나 노트북에 넣어요. 학습 결과 그래프(`results.png`)도 같이 받아 두면 발표 자료로 쓸 수 있어요.

In [ ]:
shutil.make_archive('/content/carearm_result', 'zip', '/content/runs/detect')
files.download('/content/runs/detect/carearm/weights/best.pt')
files.download('/content/carearm_result.zip')